In [1]:
import argparse
import os
import cv2
import numpy as np
from PIL import Image

In [2]:
# ==== CONFIG ====
image_dir = '/home/summer_interns/brick_kilns_data/historical_data/delhi_airshed_2024/images_backup'
label_dir = '/home/summer_interns/brick_kilns_data/historical_data/delhi_airshed_2024/labels_backup'
output_base = 'data/delhi_airshed_2024_cropped'
image_ext = '.tif'

In [3]:
os.makedirs(output_base, exist_ok=True)
for i in range(3):
    os.makedirs(os.path.join(output_base, f'class_{i}'), exist_ok=True)

In [4]:
# === Normalize coords to image dims ===
def denormalize(coords, w, h):
    return np.array([[int(x * w), int(y * h)] for x, y in coords], np.int32)


In [5]:
for label_file in os.listdir(label_dir):
    if not label_file.endswith('.txt'):
        continue

    image_file = os.path.splitext(label_file)[0] + image_ext
    image_path = os.path.join(image_dir, image_file)
    label_path = os.path.join(label_dir, label_file)

    if not os.path.exists(image_path):
        print(f"Image not found: {image_file}")
        continue

    # Load .tif image using PIL and convert to OpenCV format
    try:
        pil_image = Image.open(image_path).convert('RGB')  # force 3-channel
        image = np.array(pil_image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)     # convert to OpenCV BGR
    except Exception as e:
        print(f"Failed to load {image_file}: {e}")
        continue

    h, w = image.shape[:2]

    with open(label_path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        parts = line.strip().split()
        cls = int(parts[0])
        coords = list(map(float, parts[1:]))
        obb = [(coords[i], coords[i+1]) for i in range(0, 8, 2)]
        polygon = denormalize(obb, w, h)

        # Create mask and crop
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [polygon], 255)
        masked = cv2.bitwise_and(image, image, mask=mask)

        # Crop tight bounding box
        x, y, bw, bh = cv2.boundingRect(polygon)
        cropped = masked[y:y+bh, x:x+bw]

        # Save
        out_path = os.path.join(output_base, f'class_{cls}', f"{os.path.splitext(image_file)[0]}_{i}.png")
        cv2.imwrite(out_path, cropped)

print("✅ Cropping complete.")


✅ Cropping complete.
